# MCI Ambulance Scheduling: Minimize Patient Waiting Time

这是基于 reference #12 的简化扩展模型，按课堂 supply-chain 例子的代码结构实现。

**Scope:** optimize ambulance assignment, trip sequence and hospital choice. Travel times are given inputs; road-edge paths and hospital treatment queues are not decision variables.

**Objective:** minimize total waiting **at the scene**, from readiness after triage to pickup/loading start. This differs from the paper's treatment-completion objectives.

所有输入都是 synthetic example，尚未连接 Baltimore dashboard。所有患者在时间 0 已完成 triage；每辆救护车每趟只运送一人，可多趟往返。医院容量是整个实验期间可接收的总人数，不是可重复使用的床位。

## Sets and parameters

| Symbol | Code | Meaning |
|---|---|---|
| $p\in P$ | `P` | Patients |
| $k\in K$ | `K` | Ambulances |
| $h\in H$ | `H` | Hospitals |
| $q\in Q=\{0,\ldots,|P|-1\}$ | `Q` | Ordered trip positions for each ambulance |
| $s_p$ | `patient_site[p]` | Patient location |
| $\kappa_p$ | `priority[p]` | Triage priority; higher value = higher priority |
| $r_p$ | `ready[p]` | Patient readiness time; zero in this example |
| $e_k$ | `available[k]` | Ambulance availability time |
| $d^0_{ks}$ | `start_time[k][s]` | Initial-location-to-site travel time |
| $d_{sh}$ | `travel_time[s][h]` | Site-to-hospital travel time |
| $\bar d_{hs}$ | `return_time[h][s]` | Hospital-to-next-site travel time |
| $\ell_p$ | `loading[p]` | Patient loading time |
| $v_h$ | `handover[h]` | Hospital unloading/handover time |
| $c_h$ | `hospital_capacity[h]` | Total admission capacity |
| $g_{ph}\in\{0,1\}$ | `compatible[p][h]` | Hospital eligibility |
| $\omega_p>0$ | `weight[p]` | Objective weight; all equal to one by default |

All times are in minutes after time 0. A conservative upper bound $L$ (`horizon`) bounds every timestamp. $M=L+\max_{h,s}\bar d_{hs}$ deactivates unused timing links. The input times are assumed finite and nonnegative; unreachable road pairs must be removed/forbidden in a real-data extension.

## Decision variables

| Variable | Type | Meaning |
|---|---|---|
| $x_{pkqh}$ | Binary | Ambulance $k$ transports patient $p$ to hospital $h$ on trip $q$ |
| $u_{kq}$ | Binary | Trip $q$ of ambulance $k$ is used |
| $a_{kq}$ | Continuous, $[0,L]$ | Pickup/loading start at site |
| $b_{kq}$ | Continuous, $[0,L]$ | Hospital arrival |
| $f_{kq}$ | Continuous, $[0,L]$ | Ambulance free after handover |
| $t_p$ | Continuous, $[0,L]$ | Patient pickup/loading start |
| $w_p$ | Continuous, $[0,L]$ | Patient scene waiting time |

For concise notation only, define expressions $X_{pkq}=\sum_hx_{pkqh}$ and $Y_{hkq}=\sum_px_{pkqh}$. These are sums of existing variables, not additional decision variables.

## Objective function

$$\min\ \sum_{p\in P}\omega_p w_p.$$

With $\omega_p=1$, this minimizes total scene waiting; with a fixed number of patients it also minimizes average scene waiting. Triage is enforced by a separate hard constraint, not merely by objective weights.

## Constraints

**C1. Every patient is served exactly once.**
$$\sum_{k,q,h}x_{pkqh}=1\qquad \forall p.$$

**C2. One patient/hospital assignment per used trip.**
$$\sum_{p,h}x_{pkqh}=u_{kq}\qquad \forall k,q.$$

**C3. Trips are used consecutively.**
$$u_{k,q+1}\le u_{kq}\qquad \forall k,\ q<|Q|-1.$$

**C4. Hospital compatibility.**
$$x_{pkqh}\le g_{ph}\qquad \forall p,k,q,h.$$

**C5. Hospital admission capacity.**
$$\sum_{p,k,q}x_{pkqh}\le c_h\qquad \forall h.$$

**C6. Inactive trip timestamps are zero.**
$$0\le a_{kq},b_{kq},f_{kq}\le L u_{kq}\qquad \forall k,q.$$

**C7. First-trip timing includes initial travel.**
$$a_{k0}\ge e_ku_{k0}+\sum_{p,h}d^0_{k,s_p}x_{pk0h}\qquad \forall k.$$

**C8. Hospital arrival and ambulance release.**
$$b_{kq}=a_{kq}+\sum_{p,h}(\ell_p+d_{s_ph})x_{pkqh}\qquad \forall k,q,$$
$$f_{kq}=b_{kq}+\sum_{p,h}v_hx_{pkqh}\qquad \forall k,q.$$

**C9. A vehicle cannot perform overlapping trips.** It must complete handover and travel from the previous hospital to the next pickup site.
$$a_{k,q+1}\ge f_{kq}+\bar d_{h,s_p}-M(2-Y_{hkq}-X_{p,k,q+1})$$
$$\forall k,\ q<|Q|-1,\ h,p.$$

Only when the previous destination is $h$ AND the next patient is $p$ are both sums one, so the timing constraint is active. The return leg can go to a different site.

**C10. Link each patient's pickup time to its selected trip.**
$$a_{kq}-M(1-X_{pkq})\le t_p\le a_{kq}+M(1-X_{pkq})\qquad\forall p,k,q.$$

**C11. Readiness and waiting definition.**
$$t_p\ge r_p,\qquad w_p=t_p-r_p\qquad\forall p.$$

Waiting ends at loading start. Loading, transport and hospital queues are not counted in $w_p$.

**C12. Triage priority at the same site.**
$$t_p\le t_i\qquad\forall p,i:\ s_p=s_i,\ \kappa_p>\kappa_i.$$

This follows the same-site priority logic of reference #12, constraint (25). Because all patients are ready at time zero, ordering pickup times also orders scene waits. Different sites have no global precedence; equal-priority patients can be reordered; simultaneous pickups are allowed. **Do not apply this rule unchanged to dynamic or staggered patient arrivals.**

**Domains:** $x,u\in\{0,1\}$ and $a,b,f,t,w\in[0,L]$.

## Gurobi implementation

The cells below use the requested course style: input data → initialize model → decision variables → objective → constraints → optimize → results. Code uses zero-based indices; printed labels use one-based indices.


In [1]:
# 1. Import packages
import gurobipy as gp
from gurobipy import Model, GRB, quicksum



In [2]:
# 2. Sets and input data (replace these synthetic inputs for your experiment)
P = range(4)                    # Patients
K = range(2)                    # Ambulances
H = range(2)                    # Hospitals
S = range(2)                    # MCI sites
Q = range(len(P))               # Potential trips per ambulance; not road edges

patient_site = [0, 0, 1, 1]
priority = [2, 1, 2, 1]         # Larger number = higher triage priority
ready = [0, 0, 0, 0]            # All patients triaged and ready at time 0
available = [0, 0]              # Ambulance availability times
loading = [2, 2, 3, 2]          # Loading time for each patient
handover = [3, 4]               # Unloading/handover time at each hospital

# Ambulance starting location -> site (ambulance x site)
start_time = [[4, 9],
              [8, 3]]
# Site -> hospital travel times (site x hospital)
travel_time = [[8, 14],
               [13, 6]]
# Hospital -> next site travel times (hospital x site); may be asymmetric
return_time = [[8, 13],
               [14, 6]]

# Total available admissions over this experiment, NOT reusable treatment beds
hospital_capacity = [2, 2]
# Patient x hospital: 1 if the hospital can accept/treat this patient
compatible = [[1, 0],
              [1, 1],
              [1, 1],
              [1, 1]]

# Use equal weights to minimize total (and therefore average) scene waiting.
# Different positive weights would define a DIFFERENT, weighted objective.
weight = [1, 1, 1, 1]

# Conservative horizon: enough to serve everyone sequentially if assignments
# are feasible. Includes depot travel, loading, transport, handover and return.
horizon = max(ready) + max(available) + len(P) * (
    max(max(row) for row in start_time)
    + max(loading) + max(max(row) for row in travel_time)
    + max(handover) + max(max(row) for row in return_time)
)
# M must also deactivate an unused return leg, whose RHS can reach horizon+return.
M = horizon + max(max(row) for row in return_time)



In [3]:
# 3. Initialize the model (same structure as the course example)
env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()
m = Model("mci_ambulance_waiting", env=env)
m.setParam("MIPGap", 0)



In [4]:
# 4. Decision variables
# x[p,k,q,h] = 1 if ambulance k takes patient p to hospital h on trip q.
x = m.addVars(P, K, Q, H, vtype=GRB.BINARY, name="x")
# u[k,q] = 1 if trip q of ambulance k is used.
u = m.addVars(K, Q, vtype=GRB.BINARY, name="u")
# a[k,q]: pickup/loading START at the site; b[k,q]: hospital arrival;
# f[k,q]: ambulance becomes free after unloading/handover.
a = m.addVars(K, Q, lb=0, ub=horizon, vtype=GRB.CONTINUOUS, name="a")
b = m.addVars(K, Q, lb=0, ub=horizon, vtype=GRB.CONTINUOUS, name="b")
f = m.addVars(K, Q, lb=0, ub=horizon, vtype=GRB.CONTINUOUS, name="f")
# t[p]: pickup/loading start for patient p; w[p]: scene waiting time.
t = m.addVars(P, lb=0, ub=horizon, vtype=GRB.CONTINUOUS, name="t")
w = m.addVars(P, lb=0, ub=horizon, vtype=GRB.CONTINUOUS, name="w")



In [5]:
# 5. Objective function: minimize total patient waiting time at the scene
m.setObjective(quicksum(weight[p] * w[p] for p in P), GRB.MINIMIZE)



In [6]:
# 6. Constraints
# C1. Every patient is transported exactly once.
for p in P:
    m.addConstr(quicksum(x[p, k, q, h] for k in K for q in Q for h in H) == 1,
                f"Serve_patient_{p}")

# C2. One patient and one destination hospital per used trip.
for k in K:
    for q in Q:
        m.addConstr(quicksum(x[p, k, q, h] for p in P for h in H) == u[k, q],
                    f"Trip_assignment_{k}_{q}")

# C3. Use trips in order: no empty trip before a later used trip.
for k in K:
    for q in range(len(Q) - 1):
        m.addConstr(u[k, q + 1] <= u[k, q], f"Trip_order_{k}_{q}")

# C4. Only compatible patient-hospital assignments are allowed.
for p in P:
    for k in K:
        for q in Q:
            for h in H:
                m.addConstr(x[p, k, q, h] <= compatible[p][h],
                            f"Compatible_{p}_{k}_{q}_{h}")

# C5. Hospital total admission capacity over the experiment.
for h in H:
    m.addConstr(quicksum(x[p, k, q, h] for p in P for k in K for q in Q)
                <= hospital_capacity[h], f"Hospital_capacity_{h}")

# C6. Unused trips have zero timestamps.
for k in K:
    for q in Q:
        m.addConstr(a[k, q] <= horizon * u[k, q], f"Active_pickup_{k}_{q}")
        m.addConstr(b[k, q] <= horizon * u[k, q], f"Active_arrival_{k}_{q}")
        m.addConstr(f[k, q] <= horizon * u[k, q], f"Active_release_{k}_{q}")

# C7. Initial trip: ambulance availability + travel from its starting location.
for k in K:
    m.addConstr(a[k, 0] >= available[k] * u[k, 0]
                + quicksum(start_time[k][patient_site[p]] * x[p, k, 0, h]
                           for p in P for h in H), f"First_trip_{k}")

# C8. Patient pickup -> hospital arrival -> ambulance release.
for k in K:
    for q in Q:
        m.addConstr(b[k, q] == a[k, q]
                    + quicksum((loading[p] + travel_time[patient_site[p]][h])
                               * x[p, k, q, h] for p in P for h in H),
                    f"Hospital_arrival_{k}_{q}")
        m.addConstr(f[k, q] == b[k, q]
                    + quicksum(handover[h] * x[p, k, q, h] for p in P for h in H),
                    f"Ambulance_release_{k}_{q}")

# C9. A later pickup must allow handover AND travel from the previous hospital.
# Both binary expressions must equal 1 to activate this timing constraint.
for k in K:
    for q in range(len(Q) - 1):
        for h in H:
            for p in P:
                previous_hospital = quicksum(x[i, k, q, h] for i in P)
                next_patient = quicksum(x[p, k, q + 1, j] for j in H)
                m.addConstr(a[k, q + 1] >= f[k, q]
                            + return_time[h][patient_site[p]]
                            - M * (2 - previous_hospital - next_patient),
                            f"Next_pickup_{k}_{q}_{h}_{p}")

# C10. Link each patient's pickup time to the assigned ambulance trip.
for p in P:
    for k in K:
        for q in Q:
            assigned = quicksum(x[p, k, q, h] for h in H)
            m.addConstr(t[p] >= a[k, q] - M * (1 - assigned),
                        f"Pickup_link_lower_{p}_{k}_{q}")
            m.addConstr(t[p] <= a[k, q] + M * (1 - assigned),
                        f"Pickup_link_upper_{p}_{k}_{q}")

# C11. No pickup before patient readiness; waiting stops when loading starts.
for p in P:
    m.addConstr(t[p] >= ready[p], f"Patient_ready_{p}")
    m.addConstr(w[p] == t[p] - ready[p], f"Waiting_time_{p}")

# C12. Triage priority within the SAME site, as in reference #12.
# All ready[p] = 0 in this static example. Simultaneous pickup is allowed.
# With staggered/dynamic arrivals, redesign this rule before using the model.
for p in P:
    for i in P:
        if patient_site[p] == patient_site[i] and priority[p] > priority[i]:
            m.addConstr(t[p] <= t[i], f"Triage_priority_{p}_{i}")



In [7]:
# 7. Optimize the model
m.optimize()



In [8]:
# 8. Print results
if m.status == GRB.OPTIMAL:
    print(f"Optimal total waiting time: {m.objVal:.2f} patient-minutes")
    print(f"Average waiting time: {sum(w[p].X for p in P) / len(P):.2f} minutes")
    print("\nPatient waiting times:")
    for p in P:
        print(f"  Patient {p+1}: site {patient_site[p]+1}, priority {priority[p]}, "
              f"pickup {t[p].X:.2f}, waiting {w[p].X:.2f} minutes")
    print("\nAmbulance schedules (labels are 1-based):")
    for k in K:
        for q in Q:
            for p in P:
                for h in H:
                    if x[p, k, q, h].X > 0.5:
                        print(f"  Ambulance {k+1}, trip {q+1}: patient {p+1} "
                              f"at site {patient_site[p]+1} -> hospital {h+1}; "
                              f"pickup {a[k,q].X:.2f}, hospital arrival {b[k,q].X:.2f}, "
                              f"free {f[k,q].X:.2f}")
else:
    print(f"Gurobi status: {m.status}")


Optimal total waiting time: 54.00 patient-minutes
Average waiting time: 13.50 minutes

Patient waiting times:
  Patient 1: site 1, priority 2, pickup 4.00, waiting 4.00 minutes
  Patient 2: site 1, priority 1, pickup 25.00, waiting 25.00 minutes
  Patient 3: site 2, priority 2, pickup 3.00, waiting 3.00 minutes
  Patient 4: site 2, priority 1, pickup 22.00, waiting 22.00 minutes

Ambulance schedules (labels are 1-based):
  Ambulance 1, trip 1: patient 1 at site 1 -> hospital 1; pickup 4.00, hospital arrival 14.00, free 17.00
  Ambulance 1, trip 2: patient 2 at site 1 -> hospital 2; pickup 25.00, hospital arrival 41.00, free 45.00
  Ambulance 2, trip 1: patient 3 at site 2 -> hospital 2; pickup 3.00, hospital arrival 12.00, free 16.00
  Ambulance 2, trip 2: patient 4 at site 2 -> hospital 1; pickup 22.00, hospital arrival 37.00, free 40.00


## Interpretation and validation

Baseline optimum: **54 patient-minutes** total waiting, **13.5 minutes** average waiting. Patient pickup times are **4, 25, 3, 22**. Multiple optimal hospital assignments may exist, especially for each ambulance's final trip, because hospital travel after the final pickup does not affect the waiting-only objective.

The objective was independently checked by exhaustive enumeration of sequences, ambulance assignments, hospital assignments and the resulting vehicle/triage timing constraints. Increasing site 1 → hospital 1 travel time from 8 to 20 gives **66 patient-minutes**; reducing hospital total admissions to 2 for 4 patients is infeasible.

For a real experiment, replace the synthetic patient, ambulance, hospital and travel-time inputs together. Road-closure scenarios can supply different shortest-path time matrices. Dynamic road closures, reusable beds, treatment queues, and online arrival decisions require extensions.

**Sources:** [Repoussis et al., reference #12](../reference/12_Repoussis_Emergency_Preparedness_Published.pdf), §3 and constraint (25); user's supplied course example; [Gurobi Python Model API](https://docs.gurobi.com/projects/optimizer/en/current/reference/python/model.html). This model is an adaptation, not an exact replication of the paper.
